In [90]:
import pandas as pd
from bs4 import BeautifulSoup

In [99]:
file_url = input('Введите ссылку на файл формата tsv: ')
if file_url == '':
    file_url = 'dialogues.tsv'
data = pd.DataFrame(pd.read_csv(file_url, sep='\t'))

In [100]:
def html_to_text(x):
    soup = BeautifulSoup(x, 'html.parser')
    return soup.get_text(strip=True)

In [101]:
data['persona_1_profile'] = data['persona_1_profile'].apply(html_to_text)
data['persona_2_profile'] = data['persona_2_profile'].apply(html_to_text)
data['dialogue'] = data['dialogue'].apply(html_to_text)

In [104]:
data.iloc[0][1]

C:\Users\foksm\AppData\Local\Temp\ipykernel_23436\4249068177.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  data.iloc[0][1]


'Ищу принца.Веду активный образ жизни.Люблю читать классику.Выращиваю фиалки.Люблю общение.'

In [92]:
soup = BeautifulSoup(data.iloc[0][1], 'html.parser')
text = soup.get_text(strip=True)
text

C:\Users\foksm\AppData\Local\Temp\ipykernel_23436\2691236615.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soup = BeautifulSoup(data.iloc[0][1], 'html.parser')


'Ищу принца.Веду активный образ жизни.Люблю читать классику.Выращиваю фиалки.Люблю общение.'

In [112]:
data.shape[0]

10013

In [77]:
data.describe()

,persona_1_profile,persona_2_profile,dialogue
count,10013,10013,10013
unique,1498,1501,9811
top,<span class=participant_1>Мне 29 лет<br />Я лю...,<span class=participant_2>Я люблю пиво.<br />Я...,<span class=participant_1>Пользователь 1: прив...
freq,16,15,2


In [78]:
data = data.dropna()
data.shape

(10013, 3)

In [79]:
data.iloc[4][0]

C:\Users\foksm\AppData\Local\Temp\ipykernel_23436\3617927494.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  data.iloc[4][0]


'<span class=participant_1>Я школьница.<br />Я ещё учусь.<br />Но я мечтаю работать.<br />Я обожаю родителей.<br />И не люблю учиться.<br /></span>'

In [80]:
data['persona_1_profile'] = data['persona_1_profile'].apply(lambda x: x.replace('<br />', '.').replace('..', '.'))
data['persona_2_profile'] = data['persona_2_profile'].apply(lambda x: x.replace('<br />', '.').replace('..', '.'))
data['dialogue'] = data['dialogue'].apply(lambda x: x.replace('<br />', '.').replace('..', '.'))

In [86]:
data.iloc[1][1]

C:\Users\foksm\AppData\Local\Temp\ipykernel_23436\3822807447.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  data.iloc[1][1]


'Я бизнесмен.У меня скоро свадьба.Меня любят только за деньги.Я не люблю людей.не люблю тупые опросы.'

In [109]:
from sklearn.pipeline import make_pipeline

In [111]:
text_pipeline = make_pipeline(
    html_to_text(x)
)

NameError: name 'x' is not defined

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import os, asyncio

In [105]:
import os, asyncio
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from sentence_transformers import SentenceTransformer
import pandas as pd
from bs4 import BeautifulSoup

load_dotenv()

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
COLLECTION_USERS = os.getenv("QDRANT_COLLECTION", "users")
COLLECTION_DIALOGS = os.getenv("QDRANT_COLLECTION", "dialogs")

d:\medved_anikin_1710\find_meaning\main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [106]:
qdrant = QdrantClient(url=QDRANT_URL)
embedder = SentenceTransformer("intfloat/e5-small")

In [107]:
try:
    qdrant.get_collection(COLLECTION_USERS)
except:
    qdrant.create_collection(
        collection_name=COLLECTION_USERS,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )

In [113]:
data['id'] = data.index

In [121]:
for i in data['id'][::-1]:
    print(data.loc[i])

persona_1_profile    Я женат.Люблю свою жену.Работаю прорабом на ст...
persona_2_profile    Я люблю читатьЯ умею плаватьЯ преподаватель в ...
dialogue             Пользователь 2: Привет! рада новому знакомству...
id                                                               10012
Name: 10012, dtype: object
persona_1_profile    Моя мама живет со мнойЯ люблю есть пирогиУ мен...
persona_2_profile    Я воспитательлюблю готовитьлюблю велоспортУвле...
dialogue             Пользователь 1: Приветик)Пользователь 2: Приве...
id                                                               10011
Name: 10011, dtype: object
persona_1_profile    Я студент.Я учусь на хирурга.Я очень люблю ком...
persona_2_profile    Директор турфирмы.Люблю детей.Катаюсь на сноуб...
dialogue             Пользователь 1: ПриветПользователь 2: приветст...
id                                                               10010
Name: 10010, dtype: object
persona_1_profile    Я верная.Мне нравится уют .Я люблю посиделки у

In [ ]:
point = PointStruct(
    id=event.message.id,
    vector=emb,
    payload={"text": text, "chat_id": event.chat_id})